In [1]:
from pathlib import Path
import sys
sys.path.append(str(Path().resolve().parent / 'src'))

In [2]:
from typing import List

from data_handlers.mic_data_handler import MicDataHandler
from utils.evaluation_utils import EvaluationUtils
from utils.project_utils import ProjectUtils

import textwrap

import pandas as pd

In [3]:
project_root: Path = ProjectUtils.get_project_root(); project_root

PosixPath('/home/ssaha/Projects/redacted-text-utility')

In [4]:
mic_data_handler: MicDataHandler = MicDataHandler(project_root)

In [5]:
def evaluate_model(model_names: List[str], 
                   data_fold_values: List[int], 
                   replacement_strategies: List[str], 
                   export_path_root: Path) -> None:
    
    pe_df = mic_data_handler.get_private_entities_df()
    id_column="itemid"
    text_column="text"
    class_column="intents"
    pe_column="text_pe_ontonotes5_ner-english-ontonotes-large"
    zero_entity_retain_text=True

    for model_name in model_names:

        for k in data_fold_values:
            
            sample_size = 4339 if k in [1, 4, 5] else 4338
            data_dir_path = Path(f'/home/ssaha/model-checkpoints/mic/mltc/{model_name}/additional-embeddings-none/sample-size-{sample_size}/data-fold-{k}')
            model_file_path = data_dir_path / 'learning-rate-5e-5' / 'max-epochs-25' / 'mini-batch-size-8' / 'best-model.pt'
            test_df = mic_data_handler.get_train_dev_test_datasetdict(k=k)["test"].to_pandas()
            pe_df_test = pe_df[pe_df['itemid'].isin(test_df['itemid'])]
            entity_counts_stat = pe_df_test['pe_count_total'].describe()
            entity_counts_stat = entity_counts_stat.astype(int)
            print()
            print(f"Statistics of private entities in test samples for data_fold={k}:\n{pd.DataFrame(entity_counts_stat).to_markdown()}")
            print()
            
            input_df = test_df.copy()            
            for replacement_strategy in replacement_strategies:
                print()
                print(textwrap.dedent(f"""
                    Evaluating for configuration:
                    -> model={model_name} 
                    -> data_fold={k}
                    -> replacement_strategy={replacement_strategy}
                """).strip())
                print()
                result = EvaluationUtils.redact_and_evaluate_for_mic_mltc(input_df=input_df,
                                                                          pe_df=pe_df,
                                                                          id_column=id_column,
                                                                          text_column=text_column,
                                                                          class_column=class_column,
                                                                          pe_column=pe_column,
                                                                          replacement_strategy=replacement_strategy,
                                                                          zero_entity_retain_text=zero_entity_retain_text,
                                                                          data_dir_path=data_dir_path,
                                                                          model_file_path=model_file_path)
                
                export_path_dir = export_path_root / f'{model_name}' / f'{replacement_strategy}'
                export_path_dir.mkdir(parents=True, exist_ok=True)
                export_path = export_path_dir / f'K{k}.txt'
                with open(export_path, 'w') as f:
                    f.write(result.detailed_results)
                
                print(export_path.read_text())
                print()

In [6]:
model_names = ["microsoft--BiomedNLP-BiomedBERT-base-uncased-abstract"] # ["xlm-roberta-large", "bert-large-cased", "microsoft--BiomedNLP-BiomedBERT-base-uncased-abstract"]
data_fold_values = [1] # [1, 2, 3, 4, 5]
replacement_strategies = ["semantic_label_mask"] # ["semantic_label_mask", "random_mask", "generic_mask"]
export_path_root = Path("/home/ssaha/Projects/mic_mltc_metrics")

evaluate_model(model_names, data_fold_values, replacement_strategies, export_path_root)


Statistics of private entities in test samples for data_fold=1:
|       |   pe_count_total |
|:------|-----------------:|
| count |              105 |
| mean  |                1 |
| std   |                0 |
| min   |                1 |
| 25%   |                1 |
| 50%   |                1 |
| 75%   |                2 |
| max   |                5 |


Evaluating for configuration:
-> model=microsoft--BiomedNLP-BiomedBERT-base-uncased-abstract 
-> data_fold=1
-> replacement_strategy=semantic_label_mask



Writing chunked FastText format file...: 100%|██████████| 105/105 [00:00<00:00, 3135.11it/s]


2026-01-28 18:14:13,862 Reading data from /home/ssaha/model-checkpoints/mic/mltc/microsoft--BiomedNLP-BiomedBERT-base-uncased-abstract/additional-embeddings-none/sample-size-4339/data-fold-1
2026-01-28 18:14:13,863 Train: /home/ssaha/model-checkpoints/mic/mltc/microsoft--BiomedNLP-BiomedBERT-base-uncased-abstract/additional-embeddings-none/sample-size-4339/data-fold-1/train.txt
2026-01-28 18:14:13,863 Dev: /home/ssaha/model-checkpoints/mic/mltc/microsoft--BiomedNLP-BiomedBERT-base-uncased-abstract/additional-embeddings-none/sample-size-4339/data-fold-1/dev.txt
2026-01-28 18:14:13,864 Test: /home/ssaha/model-checkpoints/mic/mltc/microsoft--BiomedNLP-BiomedBERT-base-uncased-abstract/additional-embeddings-none/sample-size-4339/data-fold-1/test_redacted_with_semantic_label_mask.txt
2026-01-28 18:14:13,922 Initialized corpus /home/ssaha/model-checkpoints/mic/mltc/microsoft--BiomedNLP-BiomedBERT-base-uncased-abstract/additional-embeddings-none/sample-size-4339/data-fold-1 (label type name is

100%|██████████| 105/105 [00:00<00:00, 106.40it/s]



Results:
- F-score (micro) 0.7586
- F-score (macro) 0.5758
- Accuracy 0.5333

By class:
                       precision    recall  f1-score   support

            Greetings     0.9677    0.7895    0.8696        38
     Personal_History     0.7857    0.8800    0.8302        25
       Acute_Symptoms     0.6522    0.6000    0.6250        25
             Chitchat     0.9167    0.6875    0.7857        16
 Physical_Examination     0.9167    0.8462    0.8800        13
           Discussion     0.8182    0.8182    0.8182        11
           Medication     0.8182    1.0000    0.9000         9
   Diagnostic_Testing     0.8571    1.0000    0.9231         6
     Acute_Assessment     0.6250    1.0000    0.7692         5
         Reassessment     0.5000    0.7500    0.6000         4
Radiology_Examination     1.0000    0.8000    0.8889         5
  Therapeutic_History     0.5000    0.5000    0.5000         4
             Referral     0.5000    0.5000    0.5000         4
     Other_Treatments     0.